# Fast Factorized Back-Projection (FFBP) — Hierarchical Time-Domain Image Formation

**Objective**: Form a stripmap SAR image using Fast Factorized Back-Projection, a hierarchical algorithm that provides exact geometric fidelity for arbitrary collection geometries.

## Overview

**Fast Factorized Back-Projection** is a time-domain SAR image formation algorithm that:
1. Partitions the aperture into small **leaf subapertures**
2. Forms each leaf independently (direct back-projection)
3. Merges leaves into a **binary tree** via upsampling and coherent summation
4. Converts the final polar-format image to rectangular grid

Unlike PFA/RDA which approximate geometry, **FFBP is exact** — it handles:
- Non-linear flight tracks
- Wide beamwidths
- Arbitrary topography (with DEM integration)
- Squinted geometries

## Theory

### Back-Projection Fundamentals

For each image pixel $\mathbf{r}$ and each pulse $i$:
1. Compute **slant range**: $R_i = |\mathbf{r} - \mathbf{p}_i|$ (platform position $\mathbf{p}_i$)
2. Compute **round-trip time**: $\tau_i = 2R_i / c$
3. **Interpolate** signal at time $\tau_i$: $s_i(\tau_i)$
4. **Sum** over all pulses: $I(\mathbf{r}) = \sum_i s_i(\tau_i)$

Direct back-projection is $O(N_p \times N_{\text{pixels}})$ — prohibitively slow for large scenes.

### Fast Factorized Approach

FFBP reduces complexity to $O(N_p \log N_p)$ via:

1. **Leaf formation**: Partition aperture into $L$ leaves of $M$ pulses each ($M \approx 8$)
   - Each leaf forms a **coarse polar image** via direct back-projection
   - Cost: $O(L \times M \times N_{\text{pixels}})$

2. **Binary tree merge**: Recursively merge pairs of images
   - Upsample each image by 2× in polar coordinates
   - Coherently sum (phase-aligned addition)
   - Cost per level: $O(N_{\text{pixels}})$, total: $O(N_{\text{pixels}} \log L)$

3. **Polar-to-rectangular conversion**: Final interpolation to Cartesian grid

### Resolution

- **Range resolution**: $\rho_r = c / 2B$ (bandwidth $B$)
- **Cross-range resolution**: $\rho_x = \lambda / (2 \sin \theta)$ (grazing angle $\theta$)
- **Angular samples**: `n_angular` controls azimuth sampling density

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | CPHD reader via factory (`get_reader('cphd', ...)`) |
| `grdl.image_processing.sar` | `FastBackProjection` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import napari

from grdl.IO import get_reader
from grdl.image_processing.sar.image_formation import FastBackProjection

## Configuration — User Paths

In [ ]:
# Path to CPHD file (stripmap collection)
CPHD_PATH = Path("/path/to/your/stripmap_cphd.cphd")

# Validate path
assert CPHD_PATH.exists(), f"CPHD file not found: {CPHD_PATH}"
assert CPHD_PATH.suffix.lower() in [".cphd", ".cph"], f"Expected CPHD file, got {CPHD_PATH.suffix}"

print(f"✓ CPHD file: {CPHD_PATH.name}")

## Step 1: Load CPHD Metadata and Signal

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    meta = reader.metadata
    
    # Extract collection parameters
    channel = meta.data.channels[0]
    n_vectors = channel.num_vectors
    n_samples = channel.num_samples
    
    print(f"Collection mode: {meta.data.radar_mode}")
    print(f"Phase history: {n_vectors} vectors × {n_samples} samples")
    print(f"Center frequency: {meta.data.tx_frequency_nominal / 1e9:.2f} GHz")
    
    # Read full signal
    signal = reader.read_signal(channel_id=channel.identifier)

print(f"\nSignal shape: {signal.shape}, dtype: {signal.dtype}")

## Step 2: Configure FFBP Parameters

In [ ]:
# FFBP algorithm configuration
ffbp_params = {
    'leaf_size': 8,                # Pulses per leaf subaperture (2^n recommended)
    'n_angular': 128,              # Angular samples per tree node
    'range_weighting': None,       # None, 'taylor', 'hamming', 'hanning'
    'cross_range_spacing': None,   # None = auto from Nyquist, or float in meters
}

print("FFBP Configuration:")
for k, v in ffbp_params.items():
    print(f"  {k}: {v}")

## Step 3: Initialize FFBP Processor

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    ffbp = FastBackProjection(
        metadata=reader.metadata,
        **ffbp_params
    )
    
    # Query tree structure
    n_leaves = n_vectors // ffbp_params['leaf_size']
    tree_depth = int(np.ceil(np.log2(n_leaves)))
    
    print(f"FFBP initialized")
    print(f"Leaves: {n_leaves} ({ffbp_params['leaf_size']} pulses each)")
    print(f"Tree depth: {tree_depth} levels")
    print(f"Angular samples: {ffbp_params['n_angular']}")

## Step 4: Run FFBP Image Formation

The `FastBackProjection.form()` method:
1. **Leaf formation**: Direct back-projection for $M$ pulses
2. **Binary tree merge**: Hierarchical upsampling + coherent sum
3. **Polar-to-rectangular**: Final interpolation

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    ffbp = FastBackProjection(metadata=reader.metadata, **ffbp_params)
    signal = reader.read_signal()
    
    # Form the stripmap image via hierarchical back-projection
    print("Running FFBP image formation...")
    image = ffbp.form(signal)

print(f"✓ Image formed: {image.shape}, dtype: {image.dtype}")

## Step 5: Compute Image Statistics

In [ ]:
magnitude = np.abs(image)
phase = np.angle(image)

print("FFBP Image Statistics:")
print(f"  Magnitude — min: {magnitude.min():.2e}, max: {magnitude.max():.2e}")
print(f"  Magnitude — mean: {magnitude.mean():.2e}, std: {magnitude.std():.2e}")

# dB-scale magnitude
magnitude_db = 20 * np.log10(magnitude + 1e-12)
vmax = magnitude_db.max()
vmin = vmax - 50.0  # 50 dB dynamic range

print(f"  dB range: [{vmin:.1f}, {vmax:.1f}] dB")

## Step 6: Visualization — Interactive Napari Viewer

In [ ]:
viewer = napari.Viewer(title="FFBP Stripmap Image")

# Layer 1: Magnitude (dB scale)
viewer.add_image(
    magnitude_db,
    name="FFBP Magnitude (dB)",
    colormap="gray",
    contrast_limits=[vmin, vmax],
)

# Layer 2: Phase
viewer.add_image(
    phase,
    name="Phase (rad)",
    colormap="twilight",
    visible=False,
)

print("✓ Napari viewer launched")
print("  FFBP provides exact geometry (no far-field approximation)")
print("  Ideal for wide beamwidths and non-linear tracks")

## Physical Interpretation

### FFBP vs PFA/RDA

| Aspect | FFBP | PFA | RDA |
|--------|------|-----|-----|
| **Geometry** | Exact (no approximation) | Far-field approximation | Start-stop approximation |
| **Wide beamwidths** | ✅ Handles | ❌ Breaks down | ❌ Breaks down |
| **Non-linear tracks** | ✅ Exact | ❌ Linearization errors | ❌ Reference-point errors |
| **DEM integration** | ✅ Native | ⚠️ Post-processing | ⚠️ Post-processing |
| **Computational cost** | $O(N \log N)$ | $O(N \log N)$ | $O(N \log N)$ |
| **Memory** | Higher | Lower | Lowest |

### Leaf Size Selection
- **Smaller leaves** (4, 8): More tree levels, higher overhead, better geometric accuracy
- **Larger leaves** (16, 32): Fewer levels, faster, but coarser leaf images
- **Power of 2**: Optimal for binary tree (8, 16, 32, 64)

### Angular Sampling
- `n_angular`: Number of polar grid samples in azimuth (per tree node)
- Higher values → finer azimuth sampling → better resolution, slower processing
- Typical: 64–256

### When to Use FFBP
- **Non-standard geometries**: Circular SAR, arbitrary tracks
- **Wide-angle collections**: Squinted or side-looking
- **Terrain following**: When DEM is critical during formation
- **Research**: Validating PFA/RDA approximations

## Summary

Successfully demonstrated:
- ✅ FFBP hierarchical back-projection from CPHD
- ✅ Leaf formation and binary tree merging
- ✅ Exact geometric fidelity (no far-field assumption)
- ✅ Configurable leaf size and angular sampling
- ✅ Interactive napari visualization

### Key GRDL Patterns
- `get_reader('cphd', path)` for stripmap CPHD
- `FastBackProjection(metadata, leaf_size=8, n_angular=128)`
- Single `form(signal)` call handles full FFBP pipeline
- Exact geometry vs approximate frequency-domain methods

### Next Steps
- Try `leaf_size=16` for faster processing (fewer tree levels)
- Increase `n_angular=256` for finer azimuth resolution
- Compare with PFA/RDA for same CPHD file (geometric differences)
- Apply range weighting (`'taylor'`) for sidelobe suppression
- Export to SICD: `SICDWriter.write(image, metadata, output_path)`